In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

d:\Anaconda\envs\citadel\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os


%pwd
os.chdir("../")
%pwd

def load_pdf_files(data):
    loader=DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
        )
    documents=loader.load()
    return documents

extracted_data=load_pdf_files("data")

len(extracted_data)

from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

minimal_docs=filter_to_minimal_docs(extracted_data)

def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunk=text_split(minimal_docs)

In [3]:
from langchain.embeddings import HuggingFaceEmbeddings


In [5]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()


C:\Users\khush\AppData\Local\Temp\ipykernel_29744\3916692824.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


vector=embedding.embed_query("hello people!")
len(vector) #also mentioned in model card from hugging face

In [7]:
from dotenv import load_dotenv
import os
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")


os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY



In [8]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)
index_name = "stoic-chatbot"

In [9]:
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

from pinecone import Pinecone, ServerlessSpec

existing_indexes = [index.name for index in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )


index = pc.Index(index_name)


from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name,
    namespace="default"  # optional
)


from langchain_pinecone import PineconeVectorStore

docsearch=PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name
)


In [11]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})
retrieved_docs = retriever.invoke("How to be happier?")
retrieved_docs

[Document(id='518c129c-e835-4a34-8277-4b44adc06ae8', metadata={'source': 'data\\meditations.pdf'}, page_content='If you can embrace this without fear or expectation—can find\nfulfillment in what you’re doing now, as Nature intended, and in\nsuperhuman truthfulness (every word, every utterance)—then your life will\nbe happy.\nNo one can prevent that.\n 13. Doctors keep their scalpels and other instruments handy, for\nemergencies. Keep your philosophy ready too—ready to understand heaven\nand earth. In everything you do, even the smallest thing, remember the'),
 Document(id='9cd1b443-00f1-4bc3-a791-5dc500935f8e', metadata={'source': 'data\\epictus.pdf'}, page_content='it possible for a person like that to be happy? [46] Well, God has\nsent among you a person who will prove by example that it can be\ndone. [47] ‘Look at me, I have no home, no city, no property, no\nslave; I sleep on the ground; I haven’t a wife or children, no o\x00cer’s\nquarters – just earth, and sky, and one lousy cloa

In [12]:
import os
from langchain_openai import ChatOpenAI

chatModel = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="microsoft/phi-3-mini-128k-instruct",
    temperature=0.7
)

response = chatModel.invoke("What is the root cause of suffering?")
print(response.content)


 The root cause of suffering, according to Buddhist philosophy, is primarily identified as attachment or desire (Tanha). This can manifest in three forms: craving for sensory pleasures (Kamacchanda), craving for existence (Bhava), and craving for non-existence (Vedanannadhida). These cravings are driven by ignorance (Avidya) which causes individuals to perceive impermanence, unsatisfactoriness, and non-self as permanent, satisfying, and self, respectively. By understanding and ultimately overcoming these cravings and ignorance through the Eightfold Path, one can alleviate suffering, leading to the state of Nirvana, which is the extinguishing of desire and ignorance.


In [13]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a stoic-minded chatbot for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise and stoic.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])


In [14]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)


In [15]:
response = rag_chain.invoke({"input": "How can I be happier in life"})
print(response["answer"])

 To be happier, focus on finding contentment with what you have and embrace stoic principles. Cultivate gratitude for present circumstances and practice acceptance of things beyond your control. Engage in acts of kindness and maintain a positive outlook. Remember, happiness comes from within when you align your actions with virtue and reason.
